# 71 — Data Connector Playground (LlamaIndex)

## Goal
Connect **files, APIs, and databases** as data sources,
index them with LlamaIndex, and **compare retrieval quality**
across different connectors.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Data connectors** | Load data from files, APIs, DBs |
| **Document model** | LlamaIndex `Document` objects |
| **Index types** | VectorStoreIndex for embedding-based search |
| **Query engine** | Natural language Q&A over indexed data |
| **Retrieval comparison** | Compare results from different sources |

## Stack
- `llama-index-core` 0.14+
- `llama-index-llms-ollama` (local Ollama)
- `llama-index-embeddings-huggingface`
- `qwen3.5:9b` via Ollama

In [ ]:
import os, warnings, json, tempfile, sqlite3
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from pathlib import Path
from llama_index.core import (
    VectorStoreIndex, Document, Settings,
    SimpleDirectoryReader, StorageContext,
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("All imports OK")

## Step 1 — Configure LLM & Embeddings

LlamaIndex uses a global `Settings` object to configure
the **LLM** and **embedding model** used by all indexes
and query engines.

In [ ]:
# Configure Ollama as our LLM
Settings.llm = Ollama(model="qwen3.5:9b", request_timeout=120.0, temperature=0)

# Use a local HuggingFace embedding model (no API key needed)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Keep chunks small for this demo
Settings.chunk_size = 256
Settings.chunk_overlap = 50

print(f"LLM: {Settings.llm.model}")
print(f"Embed model: {Settings.embed_model.model_name}")

## Step 2 — Connector 1: Plain Text Files

The simplest connector — read `.txt` files from a directory.
`SimpleDirectoryReader` handles this automatically.

In [ ]:
# Create sample text files
DATA_DIR = Path("playground_data")
DATA_DIR.mkdir(exist_ok=True)

files = {
    "machine_learning.txt": """Machine Learning Overview
Machine learning is a subset of artificial intelligence that enables systems
to learn from data without being explicitly programmed. There are three main
types: supervised learning (labeled data), unsupervised learning (pattern
discovery), and reinforcement learning (reward-based learning).
Key algorithms include linear regression, decision trees, neural networks,
and support vector machines. Deep learning, a subset of ML, uses multi-layer
neural networks to learn hierarchical representations.""",

    "python_basics.txt": """Python Programming Basics
Python is a high-level, interpreted programming language known for its
readability. Key features include dynamic typing, garbage collection, and
extensive standard library. Python supports multiple paradigms: procedural,
object-oriented, and functional programming.
Popular frameworks: Django and Flask for web, NumPy and Pandas for data
science, PyTorch and TensorFlow for deep learning.""",

    "cloud_computing.txt": """Cloud Computing Fundamentals
Cloud computing delivers computing resources over the internet. The three
service models are: IaaS (Infrastructure), PaaS (Platform), and SaaS
(Software). Major providers: AWS, Azure, Google Cloud.
Benefits include scalability, cost efficiency, and global availability.
Containers (Docker) and orchestration (Kubernetes) are key technologies
for deploying applications in the cloud.""",
}

for name, content in files.items():
    (DATA_DIR / name).write_text(content, encoding="utf-8")

# Load with SimpleDirectoryReader
file_docs = SimpleDirectoryReader(str(DATA_DIR)).load_data()
print(f"Loaded {len(file_docs)} documents from files")
for doc in file_docs:
    print(f"  - {doc.metadata.get('file_name', 'unknown')}: {len(doc.text)} chars")

## Step 3 — Connector 2: API Data (Simulated)

In production you'd use `llama-index-readers-web` or
custom readers. Here we simulate API responses as
`Document` objects to show the pattern.

In [ ]:
# Simulate API responses (e.g., from a knowledge base API)
api_responses = [
    {
        "title": "REST API Design Principles",
        "body": """REST (Representational State Transfer) is an architectural style
for designing networked applications. Key principles: statelessness, uniform
interface, client-server separation, cacheability. HTTP methods map to CRUD:
GET (read), POST (create), PUT (update), DELETE (remove). Use proper status
codes: 200 OK, 201 Created, 404 Not Found, 500 Server Error.""",
        "source": "api/knowledge-base",
    },
    {
        "title": "Database Indexing Strategies",
        "body": """Database indexes speed up queries by creating efficient lookup
structures. B-tree indexes are the default in most RDBMS. Hash indexes are
fast for equality lookups. Full-text indexes enable text search. Composite
indexes cover multiple columns. Too many indexes slow down writes.
Always index foreign keys and frequently-queried columns.""",
        "source": "api/tech-wiki",
    },
]

# Convert API responses to LlamaIndex Documents
api_docs = [
    Document(
        text=f"{r['title']}\n{r['body']}",
        metadata={"source": r["source"], "title": r["title"]},
    )
    for r in api_responses
]

print(f"Created {len(api_docs)} documents from API data")
for doc in api_docs:
    print(f"  - [{doc.metadata['source']}] {doc.metadata['title']}")

## Step 4 — Connector 3: SQLite Database

We create a SQLite database with a `knowledge` table
and read rows into `Document` objects.

In [ ]:
# Create a small SQLite database
DB_PATH = "playground_knowledge.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS knowledge")
cur.execute("""
    CREATE TABLE knowledge (
        id INTEGER PRIMARY KEY,
        topic TEXT,
        content TEXT
    )
""")

rows = [
    ("DevOps", """DevOps combines software development and IT operations to
shorten the development lifecycle. Key practices: CI/CD pipelines, infrastructure
as code (Terraform/Pulumi), monitoring (Prometheus/Grafana), and containerization.
The goal is to deliver software faster and more reliably."""),
    ("Cybersecurity", """Cybersecurity protects systems from digital attacks.
Key areas: network security, application security, identity management.
Common threats: phishing, ransomware, SQL injection, XSS. Best practices:
least privilege access, encryption, regular patching, security audits."""),
]

cur.executemany("INSERT INTO knowledge (topic, content) VALUES (?, ?)", rows)
conn.commit()

# Read from database into Documents
cur.execute("SELECT topic, content FROM knowledge")
db_docs = [
    Document(
        text=f"{topic}\n{content}",
        metadata={"source": "sqlite", "topic": topic},
    )
    for topic, content in cur.fetchall()
]
conn.close()

print(f"Created {len(db_docs)} documents from SQLite")
for doc in db_docs:
    print(f"  - [{doc.metadata['source']}] {doc.metadata['topic']}")

## Step 5 — Build Indexes & Compare Retrieval

We build **separate indexes** for each source,
plus a **combined index** to compare retrieval quality.

In [ ]:
# Build separate indexes
file_index = VectorStoreIndex.from_documents(file_docs)
api_index = VectorStoreIndex.from_documents(api_docs)
db_index = VectorStoreIndex.from_documents(db_docs)

# Build combined index
all_docs = file_docs + api_docs + db_docs
combined_index = VectorStoreIndex.from_documents(all_docs)

print(f"Indexes built:")
print(f"  Files: {len(file_docs)} docs")
print(f"  API:   {len(api_docs)} docs")
print(f"  DB:    {len(db_docs)} docs")
print(f"  Combined: {len(all_docs)} docs")

In [ ]:
# Create query engines
file_engine = file_index.as_query_engine(similarity_top_k=2)
api_engine = api_index.as_query_engine(similarity_top_k=2)
db_engine = db_index.as_query_engine(similarity_top_k=2)
combined_engine = combined_index.as_query_engine(similarity_top_k=3)

print("Query engines ready!")

In [ ]:
# Compare retrieval across sources
query = "What are the main types of machine learning?"

print(f"Query: {query}")
print("=" * 70)

for name, engine in [("Files", file_engine), ("Combined", combined_engine)]:
    response = engine.query(query)
    print(f"\n[{name}]")
    print(f"Answer: {response.response[:300]}")
    print(f"Sources: {[n.metadata.get('file_name', n.metadata.get('source', '?')) for n in response.source_nodes]}")

In [ ]:
# Cross-source query — tests combined index advantage
query2 = "How do CI/CD pipelines and cloud computing relate?"

print(f"Query: {query2}")
print("=" * 70)

response = combined_engine.query(query2)
print(f"\nAnswer: {response.response[:500]}")
print(f"\nSources used:")
for node in response.source_nodes:
    src = node.metadata.get('file_name', node.metadata.get('source', node.metadata.get('topic', '?')))
    print(f"  - {src} (score: {node.score:.3f})")

In [ ]:
# Query that targets DB-only content
query3 = "What are common cybersecurity threats?"

print(f"Query: {query3}")
print("=" * 70)

for name, engine in [("DB only", db_engine), ("Combined", combined_engine)]:
    response = engine.query(query3)
    print(f"\n[{name}]")
    print(f"Answer: {response.response[:300]}")

## Step 6 — Retriever-Level Comparison

Instead of full Q&A, let's look at **raw retrieval**
to see which chunks each index returns.

In [ ]:
# Direct retrieval comparison
file_retriever = file_index.as_retriever(similarity_top_k=2)
combined_retriever = combined_index.as_retriever(similarity_top_k=5)

query4 = "what frameworks are used for web development?"

print(f"Query: {query4}")
print("=" * 70)

print("\n--- File Index Retrieval ---")
for node in file_retriever.retrieve(query4):
    src = node.metadata.get('file_name', '?')
    print(f"  [{node.score:.3f}] {src}: {node.text[:100]}...")

print("\n--- Combined Index Retrieval ---")
for node in combined_retriever.retrieve(query4):
    src = node.metadata.get('file_name', node.metadata.get('source', node.metadata.get('topic', '?')))
    print(f"  [{node.score:.3f}] {src}: {node.text[:100]}...")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **SimpleDirectoryReader** | Load `.txt`, `.pdf`, etc. from a folder |
| **Document objects** | Wrap text + metadata from any source |
| **API connector** | Convert API JSON → `Document(text=..., metadata=...)` |
| **DB connector** | Query SQL rows → `Document` per row |
| **VectorStoreIndex** | Embed & index documents for similarity search |
| **Query engine** | Full pipeline: retrieve → synthesize answer |
| **Retriever** | Just the retrieval step (no LLM generation) |
| **Combined index** | Merge all sources for cross-domain Q&A |

## LlamaIndex Data Loading Ecosystem
```
Data Sources                  LlamaIndex Readers              Index
──────────                   ──────────────────              ─────
Local Files (.txt, .pdf)  →  SimpleDirectoryReader     →
Web Pages                 →  SimpleWebPageReader       →   VectorStoreIndex
Databases                 →  DatabaseReader            →       ↓
APIs                      →  Custom / Document()       →   QueryEngine
Notion / Slack / etc      →  LlamaHub readers          →       ↓
                                                          Answers + Sources
```

## Extending This Notebook
- Add PDF loading with `SimpleDirectoryReader` (auto-detects file types)
- Use `DatabaseReader` from `llama-index-readers-database` for real RDBMS
- Add `SimpleWebPageReader` for web content
- Try different embedding models (e.g., `all-MiniLM-L6-v2`)
- Persist the index with `index.storage_context.persist()`